# 06 - Benchmark nhiều CLIP Model trên Flickr30K

Notebook này benchmark nhiều backbone CLIP trên cùng pipeline text-to-image retrieval đã dùng ở notebook 04.

Mục tiêu chính:

- Giữ nguyên dataset Flickr30K đã clean trong `metadata.json`.
- Giữ nguyên cách đánh giá `Recall@1`, `Recall@5`, `Recall@10`.
- Chỉ thay đổi model backbone để so sánh chất lượng retrieval.

Các model benchmark mặc định:

1. `openai/clip-vit-base-patch32` - baseline hiện tại.
2. `openai/clip-vit-base-patch16` - patch nhỏ hơn, kỳ vọng hiểu chi tiết ảnh tốt hơn.
3. `openai/clip-vit-large-patch14` - model lớn hơn, kỳ vọng Recall cao hơn nhưng tốn GPU hơn.

> Lưu ý: Notebook này chưa thêm metric mới như MRR hay Median Rank. Trọng tâm chỉ là benchmark nhiều model bằng Recall@K giống notebook trước.

## 1. Import libraries

In [1]:
import gc
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import CLIPModel, CLIPProcessor

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

d:\CITD\HK3\Python for ML\Clip_Retrieval\clip-faiss-search\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.11.0+cu128
CUDA available: True


## 2. Define paths and benchmark config

Notebook này giả định đang được chạy trong thư mục `notebooks/`, giống các notebook 01-05 hiện tại.

Nếu muốn chạy thử nhanh trước khi chạy full dataset, có thể đổi:

```python
MAX_IMAGES = 1000
```

Mặc định `MAX_IMAGES = None`, nghĩa là benchmark toàn bộ dataset đã clean.

In [2]:
PROJECT_ROOT = Path("..").resolve()
METADATA_PATH = PROJECT_ROOT / "data/processed/metadata.json"
OUTPUT_DIR = PROJECT_ROOT / "data/processed/clip_model_benchmark"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# None = chạy full dataset. Đặt 1000 nếu chỉ muốn smoke test nhanh.
MAX_IMAGES = None

# Nếu True, notebook sẽ dùng lại image embeddings/results đã lưu trước đó.
REUSE_EXISTING_EMBEDDINGS = True
REUSE_EXISTING_RESULTS = True

# Vẫn giữ đúng Recall@K từ notebook 04.
K_VALUES = (1, 5, 10)

MODEL_CONFIGS = [
    {
        "model_name": "openai/clip-vit-base-patch32",
        "short_name": "clip_b32",
        "image_batch_size": 64,
        "text_batch_size": 128,
    },
    {
        "model_name": "openai/clip-vit-base-patch16",
        "short_name": "clip_b16",
        "image_batch_size": 32,
        "text_batch_size": 128,
    },
    {
        "model_name": "openai/clip-vit-large-patch14",
        "short_name": "clip_l14",
        "image_batch_size": 16,
        "text_batch_size": 64,
    },
]

print(f"Project root: {PROJECT_ROOT}")
print(f"Metadata path: {METADATA_PATH}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Device: {DEVICE}")

Project root: D:\CITD\HK3\Python for ML\Clip_Retrieval\clip-faiss-search
Metadata path: D:\CITD\HK3\Python for ML\Clip_Retrieval\clip-faiss-search\data\processed\metadata.json
Output dir: D:\CITD\HK3\Python for ML\Clip_Retrieval\clip-faiss-search\data\processed\clip_model_benchmark
Device: cuda


## 3. Load metadata

`metadata.json` được tạo từ notebook 01 và có dạng mỗi item gồm:

```python
{
    "image_id": "...",
    "image_path": "data/raw/Images/...jpg",
    "captions": [caption_1, ..., caption_5]
}
```

In [3]:
if not METADATA_PATH.exists():
    raise FileNotFoundError(
        f"Không tìm thấy metadata: {METADATA_PATH}. "
        "Hãy chạy notebook 01_prepare_metadata.ipynb trước."
    )

with METADATA_PATH.open("r", encoding="utf-8") as f:
    metadata = json.load(f)

if MAX_IMAGES is not None:
    metadata = metadata[:MAX_IMAGES]

print(f"Number of images: {len(metadata):,}")
print("Sample metadata item:")
print(metadata[0])

required_keys = {"image_id", "image_path", "captions"}
for idx, item in enumerate(metadata):
    missing_keys = required_keys - set(item.keys())
    if missing_keys:
        raise ValueError(f"Item {idx} is missing keys: {missing_keys}")
    if len(item["captions"]) != 5:
        raise ValueError(f"Item {idx} does not have exactly 5 captions")

first_image_path = PROJECT_ROOT / metadata[0]["image_path"]
print(f"First image exists: {first_image_path.exists()} - {first_image_path}")
if not first_image_path.exists():
    raise FileNotFoundError(first_image_path)

print("Metadata schema looks valid.")

Number of images: 31,782
Sample metadata item:
{'image_id': '1000092795.jpg', 'image_path': 'data/raw/Images/1000092795.jpg', 'captions': ['Two young guys with shaggy hair look at their hands while hanging out in the yard .', 'Two young , White males are outside near many bushes .', 'Two men in green shirts are standing in a yard .', 'A man in a blue shirt standing in a garden .', 'Two friends enjoy time spent together .']}
First image exists: True - D:\CITD\HK3\Python for ML\Clip_Retrieval\clip-faiss-search\data\raw\Images\1000092795.jpg
Metadata schema looks valid.


## 4. Build caption queries and ground-truth indices

Với Flickr30K, mỗi ảnh có 5 captions.

Ý tưởng đánh giá:

- Mỗi caption là một text query.
- Ground truth của caption là index của ảnh chứa caption đó.
- Nếu ảnh đúng xuất hiện trong top K kết quả retrieve thì tính là hit cho Recall@K.

In [4]:
captions = []
gt_indices = []

for image_index, item in enumerate(metadata):
    for caption in item["captions"]:
        captions.append(caption)
        gt_indices.append(image_index)

gt_indices = np.array(gt_indices, dtype=np.int64)

print(f"Number of images: {len(metadata):,}")
print(f"Number of captions/queries: {len(captions):,}")
print("Sample captions:")
for caption in captions[:5]:
    print("-", caption)
print("Sample gt_indices:", gt_indices[:10])

Number of images: 31,782
Number of captions/queries: 158,910
Sample captions:
- Two young guys with shaggy hair look at their hands while hanging out in the yard .
- Two young , White males are outside near many bushes .
- Two men in green shirts are standing in a yard .
- A man in a blue shirt standing in a garden .
- Two friends enjoy time spent together .
Sample gt_indices: [0 0 0 0 0 1 1 1 1 1]


## 5. Helper functions

Các hàm dưới đây được viết tổng quát để dùng chung cho nhiều model.

Điểm quan trọng:

- Image embeddings và text embeddings đều được L2-normalize.
- Sau normalize, inner product tương đương cosine similarity.
- Mỗi model được cache image embeddings riêng để tránh encode lại nhiều lần.

In [5]:
def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_clip_model(model_name, device):
    model = CLIPModel.from_pretrained(model_name)
    processor = CLIPProcessor.from_pretrained(model_name)
    model = model.to(device)
    model.eval()
    return model, processor


def l2_normalize(features):
    return features / features.norm(p=2, dim=-1, keepdim=True).clamp(min=1e-12)


def get_projected_image_features(model, inputs):
    """Return projected CLIP image embeddings.

    Some environments return a BaseModelOutputWithPooling object from
    get_image_features(). To avoid that incompatibility, we explicitly call
    the vision encoder and then apply the CLIP visual projection layer.
    """
    vision_outputs = model.vision_model(pixel_values=inputs["pixel_values"])
    pooled_output = vision_outputs.pooler_output
    image_features = model.visual_projection(pooled_output)
    return l2_normalize(image_features)


def get_projected_text_features(model, inputs):
    """Return projected CLIP text embeddings.

    This mirrors CLIPModel.get_text_features(), but is more robust across
    environments because it explicitly applies the text projection layer.
    """
    text_outputs = model.text_model(
        input_ids=inputs["input_ids"],
        attention_mask=inputs.get("attention_mask"),
    )
    pooled_output = text_outputs.pooler_output
    text_features = model.text_projection(pooled_output)
    return l2_normalize(text_features)


def encode_images(model, processor, metadata_items, batch_size, device):
    all_embeddings = []

    for start in tqdm(range(0, len(metadata_items), batch_size), desc="Encoding images"):
        batch_items = metadata_items[start:start + batch_size]
        images = []

        for item in batch_items:
            image_path = PROJECT_ROOT / item["image_path"]
            with Image.open(image_path) as img:
                images.append(img.convert("RGB"))

        inputs = processor(images=images, return_tensors="pt")
        inputs = {key: value.to(device) for key, value in inputs.items()}

        with torch.no_grad():
            image_features = get_projected_image_features(model, inputs)

        all_embeddings.append(image_features.cpu().numpy().astype(np.float32))

    return np.concatenate(all_embeddings, axis=0)


def evaluate_recall_at_k_batched(
    model,
    processor,
    captions,
    gt_indices,
    image_embeddings,
    batch_size,
    device,
    k_values=(1, 5, 10),
):
    image_emb_tensor = torch.from_numpy(image_embeddings).float().to(device)
    max_k = max(k_values)
    hit_counts = {k: 0 for k in k_values}
    total = 0

    for start in tqdm(range(0, len(captions), batch_size), desc="Evaluating Recall@K"):
        end = min(start + batch_size, len(captions))
        batch_texts = captions[start:end]
        batch_gt = gt_indices[start:end]

        inputs = processor(
            text=batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
        )
        inputs = {key: value.to(device) for key, value in inputs.items()}

        with torch.no_grad():
            text_features = get_projected_text_features(model, inputs)

            scores = text_features @ image_emb_tensor.T
            topk_indices = torch.topk(scores, k=max_k, dim=1).indices.cpu().numpy()

        for k in k_values:
            hits = np.any(topk_indices[:, :k] == batch_gt[:, None], axis=1)
            hit_counts[k] += hits.sum()

        total += len(batch_texts)

    del image_emb_tensor
    clear_memory()

    return {
        f"Recall@{k}": hit_counts[k] / total
        for k in k_values
    }



## 6. Run model benchmark

Phần này là phần chính của notebook.

Với mỗi model:

1. Load model và processor.
2. Encode toàn bộ image thành embedding.
3. Dùng từng caption làm query text.
4. Tính similarity giữa text embedding và image embeddings.
5. Tính Recall@1, Recall@5, Recall@10.
6. Lưu kết quả riêng cho từng model.

In [6]:
benchmark_results = []

for config in MODEL_CONFIGS:
    model_name = config["model_name"]
    short_name = config["short_name"]
    image_batch_size = config["image_batch_size"]
    text_batch_size = config["text_batch_size"]

    model_output_dir = OUTPUT_DIR / short_name
    model_output_dir.mkdir(parents=True, exist_ok=True)

    image_embeddings_path = model_output_dir / "image_embeddings.npy"
    result_path = model_output_dir / "results.json"

    print("=" * 90)
    print(f"Benchmarking model: {model_name}")
    print(f"Short name: {short_name}")
    print(f"Image batch size: {image_batch_size}")
    print(f"Text batch size: {text_batch_size}")

    if REUSE_EXISTING_RESULTS and result_path.exists():
        with result_path.open("r", encoding="utf-8") as f:
            existing_result = json.load(f)

        same_num_images = existing_result.get("num_images") == len(metadata)
        same_num_queries = existing_result.get("num_queries") == len(captions)
        same_model = existing_result.get("model_name") == model_name

        if same_num_images and same_num_queries and same_model:
            print(f"Loading existing result from: {result_path}")
            benchmark_results.append(existing_result)
            continue
        else:
            print("Existing result does not match current config. Recomputing...")

    start_time = time.time()

    model, processor = load_clip_model(model_name, DEVICE)
    print(f"Loaded model on device: {next(model.parameters()).device}")

    if REUSE_EXISTING_EMBEDDINGS and image_embeddings_path.exists():
        print(f"Loading existing image embeddings from: {image_embeddings_path}")
        image_embeddings = np.load(image_embeddings_path)

        if image_embeddings.shape[0] != len(metadata):
            print("Existing image embeddings do not match current metadata size. Re-encoding images...")
            image_embeddings = encode_images(
                model=model,
                processor=processor,
                metadata_items=metadata,
                batch_size=image_batch_size,
                device=DEVICE,
            )
            np.save(image_embeddings_path, image_embeddings)
    else:
        image_embeddings = encode_images(
            model=model,
            processor=processor,
            metadata_items=metadata,
            batch_size=image_batch_size,
            device=DEVICE,
        )
        np.save(image_embeddings_path, image_embeddings)
        print(f"Saved image embeddings to: {image_embeddings_path}")

    print(f"Image embeddings shape: {image_embeddings.shape}")
    print(f"Image embeddings dtype: {image_embeddings.dtype}")

    recalls = evaluate_recall_at_k_batched(
        model=model,
        processor=processor,
        captions=captions,
        gt_indices=gt_indices,
        image_embeddings=image_embeddings,
        batch_size=text_batch_size,
        device=DEVICE,
        k_values=K_VALUES,
    )

    elapsed_seconds = time.time() - start_time

    result_item = {
        "model_name": model_name,
        "short_name": short_name,
        "num_images": len(metadata),
        "num_queries": len(captions),
        "embedding_dim": int(image_embeddings.shape[1]),
        "Recall@1": float(recalls["Recall@1"]),
        "Recall@5": float(recalls["Recall@5"]),
        "Recall@10": float(recalls["Recall@10"]),
        "elapsed_seconds": float(elapsed_seconds),
        "image_embeddings_path": str(image_embeddings_path),
    }

    with result_path.open("w", encoding="utf-8") as f:
        json.dump(result_item, f, indent=2, ensure_ascii=False)

    benchmark_results.append(result_item)

    print("Result:")
    print(json.dumps(result_item, indent=2, ensure_ascii=False))

    del model, processor, image_embeddings
    clear_memory()

Benchmarking model: openai/clip-vit-base-patch32
Short name: clip_b32
Image batch size: 64
Text batch size: 128


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 68886.77it/s]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded model on device: cuda:0


Encoding images: 100%|██████████| 497/497 [04:51<00:00,  1.70it/s]


Saved image embeddings to: D:\CITD\HK3\Python for ML\Clip_Retrieval\clip-faiss-search\data\processed\clip_model_benchmark\clip_b32\image_embeddings.npy
Image embeddings shape: (31782, 512)
Image embeddings dtype: float32


...

Loaded model on device: cuda:0


Encoding images: 100%|██████████| 1987/1987 [12:08<00:00,  2.73it/s]


Saved image embeddings to: D:\CITD\HK3\Python for ML\Clip_Retrieval\clip-faiss-search\data\processed\clip_model_benchmark\clip_l14\image_embeddings.npy
Image embeddings shape: (31782, 768)
Image embeddings dtype: float32


Evaluating Recall@K: 100%|██████████| 2483/2483 [02:08<00:00, 19.39it/s]


Result:
{
  "model_name": "openai/clip-vit-large-patch14",
  "short_name": "clip_l14",
  "num_images": 31782,
  "num_queries": 158910,
  "embedding_dim": 768,
  "Recall@1": 0.2800453086652822,
  "Recall@5": 0.4931596501164181,
  "Recall@10": 0.5871814234472342,
  "elapsed_seconds": 914.9868812561035,
  "image_embeddings_path": "D:\\CITD\\HK3\\Python for ML\\Clip_Retrieval\\clip-faiss-search\\data\\processed\\clip_model_benchmark\\clip_l14\\image_embeddings.npy"
}


## 7. Show comparison table

Bảng này là kết quả benchmark chính của notebook 06.

Các cột `Recall@K (%)` được nhân 100 để dễ đọc.

In [7]:
results_df = pd.DataFrame(benchmark_results)

if results_df.empty:
    raise ValueError("No benchmark results found. Please run the benchmark cell above first.")

display_df = results_df.copy()
for col in ["Recall@1", "Recall@5", "Recall@10"]:
    display_df[col] = display_df[col] * 100

# Chỉ hiển thị các cột quan trọng để tránh bảng bị rối.
display_columns = [
    "short_name",
    "model_name",
    "embedding_dim",
    "num_images",
    "num_queries",
    "Recall@1",
    "Recall@5",
    "Recall@10",
    "elapsed_seconds",
]

display_df = display_df[display_columns].sort_values("Recall@1", ascending=False)

display_df.style.format({
    "Recall@1": "{:.2f}",
    "Recall@5": "{:.2f}",
    "Recall@10": "{:.2f}",
    "elapsed_seconds": "{:.1f}",
})

,short_name,model_name,embedding_dim,num_images,num_queries,Recall@1,Recall@5,Recall@10,elapsed_seconds
2,clip_l14,openai/clip-vit-large-patch14,768,31782,158910,28.00,49.32,58.72,915.0
1,clip_b16,openai/clip-vit-base-patch16,512,31782,158910,24.70,45.07,54.59,357.1
0,clip_b32,openai/clip-vit-base-patch32,512,31782,158910,21.60,41.32,50.81,372.5


## 8. Save aggregate benchmark results

Cell này lưu kết quả tổng hợp của tất cả model để dùng cho báo cáo hoặc slide.

In [8]:
aggregate_json_path = OUTPUT_DIR / "benchmark_results.json"
aggregate_csv_path = OUTPUT_DIR / "benchmark_results.csv"

with aggregate_json_path.open("w", encoding="utf-8") as f:
    json.dump(benchmark_results, f, indent=2, ensure_ascii=False)

results_df.to_csv(aggregate_csv_path, index=False)

print(f"Saved aggregate JSON to: {aggregate_json_path}")
print(f"Saved aggregate CSV to: {aggregate_csv_path}")

Saved aggregate JSON to: D:\CITD\HK3\Python for ML\Clip_Retrieval\clip-faiss-search\data\processed\clip_model_benchmark\benchmark_results.json
Saved aggregate CSV to: D:\CITD\HK3\Python for ML\Clip_Retrieval\clip-faiss-search\data\processed\clip_model_benchmark\benchmark_results.csv
